In [ ]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

# Matrix addition using 2D blocks and threads

In [ ]:
%%cuda
#include <stdio.h>

__global__ void matrixAdd(float *a, float *b, float *c, int N)
{
    int ix = blockIdx.x * blockDim.x + threadIdx.x;
    int iy = blockIdx.y * blockDim.y + threadIdx.y;

    int idx = iy*N + ix;
    c[idx] = a[idx] + b[idx];
    printf("Thread (%d, %d) computed c[%d] = %f\n", ix, iy, idx, c[idx]);
}

int main()
{
    int N = 1024;
    int M = 1024;
    
    size_t size = N * M * sizeof(float);

    float *h_a = (float *)malloc(size);
    float *h_b = (float *)malloc(size);
    float *h_c = (float *)malloc(size);

    for (int i = 0; i < N*M; i++) {
        h_a[i] = 1.0f;
        h_b[i] = 2.0f;
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, size);
    cudaMalloc(&d_b, size);
    cudaMalloc(&d_c, size);

    cudaMemcpy(d_a, h_a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, size, cudaMemcpyHostToDevice);

    dim3 blockSize(16, 16); // 16x16 threads per block
    dim3 gridSize((N + blockSize.x - 1) / blockSize.x, (M + blockSize.y - 1) / blockSize.y); // Adjusted for M
    matrixAdd<<<gridSize, blockSize>>>(d_a, d_b, d_c, N);

    cudaMemcpy(h_c, d_c, size, cudaMemcpyDeviceToHost);

    printf("Result: %f\n", h_c[0]);

    free(h_a);
    free(h_b);
    free(h_c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}